# Experiment 02 — Deterministic Baseline

**Goal:** Establish a clean, leakage-free baseline using a **BiLSTM**
model with univariate load input.

| Parameter | Value |
|-----------|-------|
| Model | BiLSTM (64 units) |
| Features | Univariate (load only) |
| Lookback | 24 h |
| Horizon | 24 h |
| Loss | MSE |
| Optimizer | Adam |
| Early stopping | val_loss, patience=15 |
| Seed | 42 |

Metrics are computed **after inverse transformation** to the original MW scale.

In [ ]:
# ── Cell 1: Environment Setup ────────────────────────────────────
from pathlib import Path
import subprocess, sys

PROJECT_ROOT = Path("/kaggle/working/stlf-entso-2026")

if not PROJECT_ROOT.exists():
    subprocess.run(
        ["git", "clone",
         "https://github.com/AlvinHarist/stlf-entso-2026.git",
         str(PROJECT_ROOT)],
        check=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
# ── Cell 2: Imports & Versions ───────────────────────────────────
import platform, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import tensorflow as tf
import sklearn

print(f"Python       : {platform.python_version()}")
print(f"TensorFlow   : {tf.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Scikit-learn : {sklearn.__version__}")

In [ ]:
# ── Cell 3: Configuration ────────────────────────────────────────
CONFIG_PATH = PROJECT_ROOT / "configs" / "baseline.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

SEED       = config["seed"]
LOOKBACK   = config["windowing"]["lookback"]
HORIZON    = config["windowing"]["horizon"]
TARGET_COL = config["data"]["target_col"]
UNITS      = config["model"]["units"]
DROPOUT    = config["model"]["dropout"]
LR         = config["model"]["learning_rate"]
EPOCHS     = config["training"]["epochs"]
BATCH_SIZE = config["training"]["batch_size"]
PATIENCE   = config["training"]["patience"]

# Kaggle data path
KAGGLE_DATA_DIR = Path("/kaggle/input/stlf-entso-2026")
DATA_PATH = None
if KAGGLE_DATA_DIR.exists():
    for p in KAGGLE_DATA_DIR.rglob("*.csv"):
        if "combined_AT" in p.name:
            DATA_PATH = p
            break
if DATA_PATH is None:
    fallback = PROJECT_ROOT / config["data"]["path"]
    if fallback.exists():
        DATA_PATH = fallback
if DATA_PATH is None:
    fallback = PROJECT_ROOT / "df_combined_AT.csv"
    if fallback.exists():
        DATA_PATH = fallback
if DATA_PATH is None:
    raise FileNotFoundError("Cannot locate df_combined_AT.csv")

RESULTS_DIR = PROJECT_ROOT / "results" / "deterministic"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset : {DATA_PATH}")
print(f"Results : {RESULTS_DIR}")
print(f"Seed    : {SEED}")

In [ ]:
# ── Cell 4: Set Seed ─────────────────────────────────────────────
from src.utils.seed import set_seed
set_seed(SEED)
print(f"Seed set to {SEED}")

In [ ]:
# ── Cell 5: Load & Split Data ────────────────────────────────────
from src.data.load_data import load_dataset
from src.data.preprocessing import chronological_split

df = load_dataset(DATA_PATH, timestamp_col=config["data"]["timestamp_col"])
train_df, val_df, test_df = chronological_split(
    df,
    train_ratio=config["split"]["train_ratio"],
    val_ratio=config["split"]["val_ratio"],
)

In [ ]:
# ── Cell 6: Preprocessing (train-only fit) ───────────────────────
from src.data.preprocessing import fit_preprocessor, transform_data

# Univariate: only the target column as feature
feature_cols = [TARGET_COL]

preprocessor = fit_preprocessor(
    train_df,
    target_col=TARGET_COL,
    feature_cols=feature_cols,
    use_yeojohnson=False,  # no skewed weather cols in univariate
)

X_train_s, y_train_s = transform_data(train_df, preprocessor)
X_val_s,   y_val_s   = transform_data(val_df,   preprocessor)
X_test_s,  y_test_s  = transform_data(test_df,  preprocessor)

print(f"Scaled shapes — Train X: {X_train_s.shape}, Val X: {X_val_s.shape}, Test X: {X_test_s.shape}")

In [ ]:
# ── Cell 7: Create Windows ───────────────────────────────────────
from src.data.windowing import create_train_windows, create_evaluation_windows

Xw_train, yw_train = create_train_windows(X_train_s, y_train_s, LOOKBACK, HORIZON)
Xw_val,   yw_val   = create_evaluation_windows(X_val_s, y_val_s, X_train_s, y_train_s, LOOKBACK, HORIZON)
Xw_test,  yw_test  = create_evaluation_windows(X_test_s, y_test_s, X_val_s, y_val_s, LOOKBACK, HORIZON)

print(f"Windows — Train: {Xw_train.shape}, Val: {Xw_val.shape}, Test: {Xw_test.shape}")

In [ ]:
# ── Cell 8: Build Model ──────────────────────────────────────────
from src.models.bilstm import build_bilstm

n_features = Xw_train.shape[2]

model = build_bilstm(
    lookback=LOOKBACK,
    n_features=n_features,
    horizon=HORIZON,
    units=UNITS,
    dropout=DROPOUT,
    learning_rate=LR,
)
model.summary()

In [ ]:
# ── Cell 9: Train ────────────────────────────────────────────────
from src.training.trainer import train_model

train_result = train_model(
    model,
    Xw_train, yw_train,
    Xw_val, yw_val,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    patience=PATIENCE,
    save_dir=RESULTS_DIR,
)

In [ ]:
# ── Cell 10: Predict & Inverse Transform ─────────────────────────
from src.data.preprocessing import inverse_y

pred_test_scaled = model.predict(Xw_test)

# Inverse transform to original MW scale
pred_test_mw = inverse_y(pred_test_scaled, preprocessor)
actual_test_mw = inverse_y(yw_test, preprocessor)

print(f"Predictions shape: {pred_test_mw.shape}")
print(f"Sample prediction (first window): {pred_test_mw[0, :5].round(1)} MW")
print(f"Sample actual     (first window): {actual_test_mw[0, :5].round(1)} MW")

In [ ]:
# ── Cell 11: Evaluate (Point Metrics) ────────────────────────────
from src.evaluation.point_metrics import compute_all_metrics, horizon_wise_metrics

metrics = compute_all_metrics(actual_test_mw, pred_test_mw)
print("\nTest Metrics (original MW scale):")
for k, v in metrics.items():
    print(f"  {k:8s}: {v:.4f}")

hw = horizon_wise_metrics(actual_test_mw, pred_test_mw)
print(f"\nHorizon-wise MAPE (h=1..5): {[f'{v:.2f}%' for v in hw['mape'][:5]]}")

In [ ]:
# ── Cell 12: Naive Baselines ─────────────────────────────────────
from src.evaluation.point_metrics import compute_naive_baselines

# Reconstruct full series in original scale for baseline lookup
full_series_mw = df[TARGET_COL].values
test_start_idx = len(train_df) + len(val_df)

baselines = compute_naive_baselines(
    actual_test_mw, full_series_mw, test_start_idx, HORIZON
)

print("\nNaive Baseline Comparison:")
print(f"  {'Model':<20s} {'MAE':>10s} {'RMSE':>10s} {'MAPE':>10s} {'sMAPE':>10s}")
print("  " + "-" * 62)
for name, m in baselines.items():
    print(f"  {name:<20s} {m['MAE']:>10.2f} {m['RMSE']:>10.2f} {m['MAPE']:>10.2f} {m['sMAPE']:>10.2f}")
print(f"  {'BiLSTM (ours)':<20s} {metrics['MAE']:>10.2f} {metrics['RMSE']:>10.2f} {metrics['MAPE']:>10.2f} {metrics['sMAPE']:>10.2f}")

In [ ]:
# ── Cell 13: Visualisation ───────────────────────────────────────
# Training loss curve
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_result["history"]["loss"], label="Train")
axes[0].plot(train_result["history"]["val_loss"], label="Validation")
axes[0].axvline(train_result["best_epoch"] - 1, color="red", linestyle="--", alpha=0.5, label=f"Best epoch ({train_result['best_epoch']})")
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Horizon-wise MAPE
axes[1].bar(range(1, HORIZON + 1), hw["mape"])
axes[1].set_title("MAPE by Forecast Horizon")
axes[1].set_xlabel("Horizon (h)")
axes[1].set_ylabel("MAPE (%)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
fig.savefig(RESULTS_DIR / "bilstm_uni_24h_diagnostics.png", dpi=150)
plt.show()

# Forecast vs Actual (first 7 days)
n_plot = min(168, len(actual_test_mw))  # ~7 days
fig2, ax2 = plt.subplots(figsize=(14, 4))
ax2.plot(actual_test_mw[:n_plot, 0], label="Actual", linewidth=0.8)
ax2.plot(pred_test_mw[:n_plot, 0], label="Predicted (h=1)", linewidth=0.8, alpha=0.8)
ax2.set_title("Forecast vs Actual — First 7 Days of Test Set (h=1)")
ax2.set_xlabel("Window index")
ax2.set_ylabel("Load (MW)")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
fig2.savefig(RESULTS_DIR / "bilstm_uni_24h_forecast.png", dpi=150)
plt.show()

In [ ]:
# ── Cell 14: Save Results ────────────────────────────────────────
result_record = {
    "experiment": "02_deterministic_baseline",
    "model": "BiLSTM",
    "features": "univariate",
    "lookback": LOOKBACK,
    "horizon": HORIZON,
    "seed": SEED,
    "units": UNITS,
    "dropout": DROPOUT,
    "learning_rate": LR,
    "epochs_trained": len(train_result["history"]["loss"]),
    "best_epoch": train_result["best_epoch"],
    "best_val_loss": train_result["best_val_loss"],
    "test_metrics": metrics,
    "baselines": baselines,
    "python_version": platform.python_version(),
    "tensorflow_version": tf.__version__,
    "numpy_version": np.__version__,
}

out_path = RESULTS_DIR / f"bilstm_uni_{HORIZON}h_seed{SEED}.json"
with open(out_path, "w") as f:
    json.dump(result_record, f, indent=2)

print(f"Results saved to {out_path}")